# pythtb gpu tests

rough. kaggle T4. kane-mele from the tutorials, supercells to make it bigger, 31x31 k mesh
needs gpu + internet on

In [ ]:
!pip install -q git+https://github.com/JoeyBryson/pythtb_gpu_accelaration_project.git

In [ ]:
import time
import numpy as np
import cupy as cp
import torch
from cupyx.profiler import benchmark
from pythtb import WFArray, Mesh
from pythtb.models import kane_mele

def bench(fn, *args, n=3):
    r = benchmark(fn, args, n_repeat=n, n_warmup=1)
    return max(r.cpu_times.mean(), r.gpu_times.mean()) * 1000   # ms

NK = 31
SIZES = [2, 4, 6, 8]
base = kane_mele(1.0, 1.0, 0.3, 0.25)
mesh = Mesh(["k", "k"]); mesh.build_grid(shape=(NK, NK), gamma_centered=True)

def model_of(n):
    return base.make_supercell([[n, 0], [0, n]]) if n > 1 else base

## 1. profile on cpu
solve = diagonalisation, berry_phase = wilson loops

In [ ]:
for n in [1] + SIZES:
    m = model_of(n)
    wfa = WFArray(m.lattice, mesh, spinful=True)
    t0 = time.perf_counter(); wfa.solve_model(model=m); t1 = time.perf_counter()
    wfa.berry_phase(axis_idx=1, state_idx=list(range(m.norb)), contin=True, berry_evals=True); t2 = time.perf_counter()
    print(f"{n}x{n}  matrix {2*m.norb:>4}   solve {t1-t0:6.2f}s   wilson {t2-t1:6.2f}s   ({100*(t2-t1)/(t2-t0):.0f}% wilson)")

```
1x1  matrix 4     solve  0.08s  wilson  0.03s
2x2  matrix 16    solve  0.14s  wilson  0.04s
4x4  matrix 64    solve  1.76s  wilson  0.61s
6x6  matrix 144   solve  9.27s  wilson  4.13s
8x8  matrix 256   solve 31.51s  wilson 13.48s
```
~70% diag / 30% wilson. use_tensorflow=True slower until ~6x6, ~1.5x at 8x8. wilson has no gpu option

## 2. eigh: numpy vs cupy vs torch
pythtb already does one batched np eigh so thats the baseline. c128 double, c64 single

In [ ]:
for n in SIZES:
    H = model_of(n).hamiltonian(mesh.flat, flatten_spin_axis=True)   # (961, dim, dim)
    t_np = bench(np.linalg.eigh, H, n=2)

    Hg = cp.asarray(H)
    t_cp128 = bench(cp.linalg.eigh, Hg)
    t_cp64 = bench(cp.linalg.eigh, Hg.astype(cp.complex64))

    Ht = torch.as_tensor(H, device="cuda")
    def teigh(X): torch.linalg.eigh(X); torch.cuda.synchronize()
    teigh(Ht); teigh(Ht.to(torch.complex64))
    t0 = time.perf_counter(); teigh(Ht); t_t128 = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter(); teigh(Ht.to(torch.complex64)); t_t64 = (time.perf_counter() - t0) * 1000

    err64 = np.abs(cp.asnumpy(cp.linalg.eigh(Hg.astype(cp.complex64))[0]) - np.linalg.eigh(H)[0]).max()
    print(f"{n}x{n}  numpy {t_np:8.0f}ms | cupy c128 {t_cp128:8.0f} c64 {t_cp64:7.0f} | torch c128 {t_t128:8.0f} c64 {t_t64:7.0f} | c64 eig err {err64:.1e}")
    del Hg, Ht; cp.get_default_memory_pool().free_all_blocks(); torch.cuda.empty_cache()

```
size  numpy   cupy128  cupy64  torch128  torch64  (ms)   c64 err
2x2      69        40     0.8        30        5          1e-05
4x4    1347      1150     117      3985      712          5e-05
6x6    7141     22522    1196      7358     1408          3e-04
8x8   24608     44521    4469     13256     2504          4e-04
```
double on T4 is bad, cupy128 slower than numpy for big ones. cupy falls off as matrices grow, torch doesnt as much
-> use torch. 8x8: ~1.9x double, ~10x single. copy to gpu ~0.26s, fine

## 3. wilson loop
batched over all k lines instead of looping. svd only used for V @ Wh so swapped it for newton-schulz (just matmuls).
product along each line done pairwise instead of one at a time

In [ ]:
def polar_svd(O):
    xp = cp.get_array_module(O)
    V, _, Wh = xp.linalg.svd(O, full_matrices=False)
    return V @ Wh

def polar_ns(O, iters=8):
    xp = cp.get_array_module(O)
    I = xp.eye(O.shape[-1], dtype=O.dtype)
    X = O
    for _ in range(iters):
        X = 0.5 * X @ (3 * I - X.conj().swapaxes(-1, -2) @ X)
    return X

def tree_product(U):
    xp = cp.get_array_module(U)
    while U.shape[-3] > 1:
        K = U.shape[-3]
        P = U[..., 0:K-1:2, :, :] @ U[..., 1:K:2, :, :]
        if K % 2:
            P = xp.concatenate([P, U[..., K-1:K, :, :]], axis=-3)
        U = P
    return U[..., 0, :, :]

def wilson_batched(u, polar):
    O = u[:, :-1].conj() @ u[:, 1:].swapaxes(-1, -2)
    return tree_product(polar(O))

def wilson_loop_version(u):
    # how pythtb does it now: loop over lines, multiply along each one at a time
    out = []
    for l in range(u.shape[0]):
        O = u[l, :-1].conj() @ u[l, 1:].swapaxes(-1, -2)
        U = polar_svd(O)
        W = U[0]
        for k in range(1, U.shape[0]):
            W = W @ U[k]
        out.append(W)
    return np.stack(out)

for n in SIZES:
    m = model_of(n)
    wfa = WFArray(m.lattice, mesh, spinful=True); wfa.solve_model(model=m)
    u = np.moveaxis(wfa.states(state_idx=list(range(m.norb)), flatten_spin_axis=True), 1, 0)
    ug = cp.asarray(u)
    t_loop = bench(wilson_loop_version, u, n=1)
    t_cpu = bench(wilson_batched, u, polar_svd, n=2)
    t_svd = bench(wilson_batched, ug, polar_svd)
    t_ns = bench(wilson_batched, ug, polar_ns)
    t_ns64 = bench(wilson_batched, ug.astype(cp.complex64), polar_ns)
    assert np.allclose(cp.asnumpy(wilson_batched(ug, polar_ns)), wilson_loop_version(u), atol=1e-8)   # same answer
    print(f"{n}x{n}  cpu loop {t_loop:7.0f}ms  cpu batched {t_cpu:7.0f}  gpu svd {t_svd:7.0f}  gpu NS {t_ns:6.0f}  gpu NS c64 {t_ns64:5.0f}")
    del ug; cp.get_default_memory_pool().free_all_blocks()

```
size  cpu loop  cpu batch  gpu svd  gpu NS  gpu NS c64  (ms)
2x2        26         22       77       7         2.5
4x4       356        391       83      26         4.4
6x6      2734       2774     8010     547          56
8x8      8570       8677    15163    1327         148
```
gpu svd slower than cpu?? NS fixes it. 8x8: ~6.5x double, ~58x single. same answers as the loop version

## rough total 8x8 (just adding the bits up, not one run)
cpu ~42s. gpu double ~15s (~2.5-3x). gpu single ~3-4s (~10x)

todo
- single precision ok? cell below compares wannier centres (cpu only). quick run up to 4x4 looked fine, ~1e-7
- what gpus on sat? T4 = worst case for double
- next bottleneck prob building H on cpu + copies

In [ ]:
# wannier centres double vs single (cpu, few mins)
import contextlib
_orig_eigh = np.linalg.eigh

def ns_wilson_c64(wfs_loop, wilson_evals=False):
    w = wfs_loop.astype(np.complex64)
    O = w[:-1].conj() @ w[1:].swapaxes(-2, -1)
    U = polar_ns(O)
    W = U[0]
    for i in range(1, len(U)):
        W = W @ U[i]
    W = W.astype(np.complex128)
    return (W, np.linalg.eigvals(W)) if wilson_evals else W

@contextlib.contextmanager
def single_precision():
    def eigh_c64(H, *a, **k):
        vals, vecs = _orig_eigh(np.asarray(H).astype(np.complex64), *a, **k)
        return vals.astype(np.float64), vecs.astype(np.complex128)
    orig_w = WFArray._wilson_loop
    np.linalg.eigh = eigh_c64
    WFArray._wilson_loop = staticmethod(ns_wilson_c64)
    try:
        yield
    finally:
        np.linalg.eigh = _orig_eigh
        WFArray._wilson_loop = orig_w

def wannier_centres(m):
    wfa = WFArray(m.lattice, mesh, spinful=True); wfa.solve_model(model=m)
    return wfa.berry_phase(axis_idx=1, state_idx=list(range(m.norb)), contin=True, berry_evals=True) / (2 * np.pi)

for n in [1, 2, 4, 6]:
    m = model_of(n)
    ref = wannier_centres(m)
    with single_precision():
        low = wannier_centres(m)
    d = np.abs(ref - low); d = np.minimum(d, 1 - d)   # centres are periodic
    print(f"{n}x{n}  max wannier centre diff {d.max():.1e}   median {np.median(d):.1e}  (units of lattice constant)")